# Jet flavour content

2D histograms of the per-jet flavour content (x: u, d, s, c, b; y: net flavour count) built from the
`<level>_<alg>_jets_<q>_content` histograms written by `jet-flv-test-histmaker.py`.
One plot per sample and jet definition (`hadron`/`parton` × `ifn`/`net`, plus `parton_getflav`), saved as
`plots/<level>_<alg>/<sample>.pdf`, plus `plots/<level>_<alg>/all_samples_<level>_<alg>.pdf` with all samples
(one page each).

* Hadron level: y in {-2, ..., 2}; parton level: y in {-1, 0, 1}.
* `parton_getflav`: the same parton-level ee_kt jets, labelled with `JetTaggingUtils::get_flavour` (heaviest
  status-2 quark within 0.3 rad, else gluon or unmatched) for comparison. This label has no sign, so only 0 and +1
  are populated. The fractions of jets labelled gluon (21) or unmatched (0) are printed.
* Each column is normalised to **all** jets of that flavour histogram (including under/overflow), so the column
  sum is 1 minus the dropped fraction, which is printed for every plot.

In [1]:
import os
import glob
import ROOT

ROOT.gROOT.SetBatch(True)
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPaintTextFormat(".3f")
ROOT.gStyle.SetPalette(ROOT.kBird)

INPUT_DIR = os.path.join("..", "..", "..", "outputs", "jet-test")
PLOT_DIR = os.path.join(INPUT_DIR, "plots")
os.makedirs(PLOT_DIR, exist_ok=True)

FLAVOURS = ["u", "d", "s", "c", "b"]
# (level, algo, max |content| shown)
JET_DEFS = [
    ("hadron", "ifn", 2), ("hadron", "net", 2),
    ("parton", "ifn", 1), ("parton", "net", 1),
    ("parton", "getflav", 1),  # JetTaggingUtils::get_flavour on the parton-level jets
]

files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.root")))
files

['../../../outputs/jet-test/wzp6_ee_nunuH_Hbb_ecm240.root',
 '../../../outputs/jet-test/wzp6_ee_nunuH_Hcc_ecm240.root',
 '../../../outputs/jet-test/wzp6_ee_nunuH_Hdd_ecm240.root',
 '../../../outputs/jet-test/wzp6_ee_nunuH_Hgg_ecm240.root',
 '../../../outputs/jet-test/wzp6_ee_nunuH_Hss_ecm240.root',
 '../../../outputs/jet-test/wzp6_ee_nunuH_Huu_ecm240.root']

In [2]:
def make_content_2d(tfile, level, algo, ymax):
    """Build a column-normalised 2D histogram (flavour vs. content) from the 1D content histograms.

    Returns the histogram and the dropped fraction (|content| > ymax) per flavour.
    """
    name = f"{os.path.basename(tfile.GetName())[:-5]}_{level}_{algo}"
    h2 = ROOT.TH2D(name, "", len(FLAVOURS), 0, len(FLAVOURS), 2 * ymax + 1, -ymax, ymax + 1)
    h2.SetDirectory(0)
    dropped = {}
    for ix, q in enumerate(FLAVOURS, start=1):
        h1 = tfile.Get(f"{level}_{algo}_jets_{q}_content")
        total = h1.Integral(0, h1.GetNbinsX() + 1)
        shown = 0.
        for content in range(-ymax, ymax + 1):
            # Content histograms have unit-width bins starting at integer edges.
            n = h1.GetBinContent(h1.FindFixBin(content + 0.5))
            h2.SetBinContent(ix, content + ymax + 1, n / total)
            shown += n
        dropped[q] = 1. - shown / total
        h2.GetXaxis().SetBinLabel(ix, q)
    for iy in range(1, h2.GetNbinsY() + 1):
        h2.GetYaxis().SetBinLabel(iy, f"{iy - ymax - 1:+d}" if iy != ymax + 1 else "0")
    return h2, dropped

In [3]:
def draw(h2, title, subdir, outname):
    c = ROOT.TCanvas(f"{subdir}_{outname}", "", 700, 550)
    c.SetRightMargin(0.15)
    h2.SetTitle(f"{title};jet flavour;flavour content")
    h2.GetZaxis().SetTitle("fraction of jets")
    h2.GetZaxis().SetRangeUser(0, 1)
    h2.GetXaxis().SetLabelSize(0.06)
    h2.GetYaxis().SetLabelSize(0.05)
    h2.SetMarkerSize(1.2)
    h2.Draw("colz text")
    os.makedirs(os.path.join(PLOT_DIR, subdir), exist_ok=True)
    c.SaveAs(os.path.join(PLOT_DIR, subdir, outname + ".pdf"))
    c.Draw()
    return c

In [4]:
canvases = {}  # jet definition -> canvases, also keeps them alive so the plots stay visible
for fname in files:
    sample = os.path.basename(fname)[:-5]
    tfile = ROOT.TFile.Open(fname)
    for level, algo, ymax in JET_DEFS:
        h2, dropped = make_content_2d(tfile, level, algo, ymax)
        print(f"{sample} {level} {algo}: dropped (|content| > {ymax}) "
              + ", ".join(f"{q}: {d:.2e}" for q, d in dropped.items()))
        subdir = f"{level}_{algo}"
        canvases.setdefault(subdir, []).append(
            draw(h2, f"{sample}, {level} level, {algo}", subdir, sample))
    h_lab = tfile.Get("parton_getflav_jets_flavour")
    n_jets = h_lab.Integral(0, h_lab.GetNbinsX() + 1)
    print(f"{sample} parton getflav: gluon {h_lab.GetBinContent(h_lab.FindFixBin(21.5)) / n_jets:.3f}, "
          f"unmatched {h_lab.GetBinContent(h_lab.FindFixBin(0.5)) / n_jets:.3f}")
    tfile.Close()

Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/hadron_ifn/wzp6_ee_nunuH_Hbb_ecm240.pdf has been created


wzp6_ee_nunuH_Hbb_ecm240 hadron ifn: dropped (|content| > 2) u: 2.20e-04, d: 3.00e-04, s: 1.00e-05, c: 0.00e+00, b: 0.00e+00
wzp6_ee_nunuH_Hbb_ecm240 hadron net: dropped (|content| > 2) u: 5.58e-03, d: 5.21e-03, s: 1.30e-04, c: 0.00e+00, b: 0.00e+00
wzp6_ee_nunuH_Hbb_ecm240 parton ifn: dropped (|content| > 1) u: 0.00e+00, d: 0.00e+00, s: 0.00e+00, c: 0.00e+00, b: 0.00e+00
wzp6_ee_nunuH_Hbb_ecm240 parton net: dropped (|content| > 1) u: 0.00e+00, d: 0.00e+00, s: 0.00e+00, c: 0.00e+00, b: 4.00e-05
wzp6_ee_nunuH_Hbb_ecm240 parton getflav: dropped (|content| > 1) u: 0.00e+00, d: 0.00e+00, s: 0.00e+00, c: 0.00e+00, b: 0.00e+00
wzp6_ee_nunuH_Hbb_ecm240 parton getflav: gluon 0.016, unmatched 0.024
wzp6_ee_nunuH_Hcc_ecm240 hadron ifn: dropped (|content| > 2) u: 2.70e-04, d: 3.20e-04, s: 0.00e+00, c: 0.00e+00, b: 0.00e+00
wzp6_ee_nunuH_Hcc_ecm240 hadron net: dropped (|content| > 2) u: 5.05e-03, d: 4.90e-03, s: 1.20e-04, c: 0.00e+00, b: 0.00e+00
wzp6_ee_nunuH_Hcc_ecm240 parton ifn: dropped (|cont

Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/hadron_net/wzp6_ee_nunuH_Hbb_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_ifn/wzp6_ee_nunuH_Hbb_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_net/wzp6_ee_nunuH_Hbb_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_getflav/wzp6_ee_nunuH_Hbb_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/hadron_ifn/wzp6_ee_nunuH_Hcc_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/hadron_net/wzp6_ee_nunuH_Hcc_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_ifn/wzp6_ee_nunuH_Hcc_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_net/wzp6_ee_nunuH_Hcc_ecm240.pdf has been created
Info

In [5]:
# One multi-page PDF per jet definition with all samples
for subdir, cs in canvases.items():
    combined = os.path.join(PLOT_DIR, subdir, f"all_samples_{subdir}.pdf")
    cs[0].Print(combined + "[")
    for c in cs:
        c.Print(combined, f"Title:{c.GetName()}")
    cs[-1].Print(combined + "]")

Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/hadron_ifn/all_samples_hadron_ifn.pdf has been created
Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/hadron_ifn/all_samples_hadron_ifn.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/hadron_ifn/all_samples_hadron_ifn.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/hadron_ifn/all_samples_hadron_ifn.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/hadron_ifn/all_samples_hadron_ifn.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/hadron_ifn/all_samples_hadron_ifn.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/hadron_ifn/all_samples_hadron_ifn.pdf
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/hadron_ifn/all_samples_hadron_ifn.pdf

Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/hadron_net/all_samples_hadron_net.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/hadron_net/all_samples_hadron_net.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/hadron_net/all_samples_hadron_net.pdf
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/hadron_net/all_samples_hadron_net.pdf has been closed
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_ifn/all_samples_parton_ifn.pdf has been created
Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/parton_ifn/all_samples_parton_ifn.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/parton_ifn/all_samples_parton_ifn.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/parton_ifn/all_sample